# Outlier Detection Model Optimization and Evaluation

Evaluate `contamination` values for both models against the labeled validation set.  
Goal: find the `contamination` that maximises F1-score on `val.csv`.

In [16]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import f1_score, precision_score, recall_score

# ── Environment detection: Colab vs. local ────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path("/content/drive/MyDrive/DTSC 5082- Group 12 - Guardian Recruit")
    DATA_DIR   = DRIVE_ROOT / "data" / "preprocessed"
    SRC_DIR    = DRIVE_ROOT / "src"
    IN_COLAB   = True

except ModuleNotFoundError:
    # Running locally — walk up from cwd until we find the repo root marker
    def _find_repo_root(marker="pytest.ini"):
        path = Path.cwd()
        while path != path.parent:
            if (path / marker).exists():
                return path
            path = path.parent
        raise FileNotFoundError(f"Cannot find repo root (looked for '{marker}')")

    REPO_ROOT  = _find_repo_root()
    DRIVE_ROOT = REPO_ROOT
    DATA_DIR   = REPO_ROOT / "data" / "processed"
    SRC_DIR    = REPO_ROOT / "src"
    IN_COLAB   = False

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

TRAIN_PATH = DATA_DIR / "train_clean_v1.csv"
VAL_PATH   = DATA_DIR / "val.csv"

print(f"Environment : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Train       : {TRAIN_PATH}")
print(f"Val         : {VAL_PATH}")

Environment : Local
Train       : /Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/data/processed/train_clean_v1.csv
Val         : /Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/data/processed/val.csv


## 2. Load Training Data & Build Feature Matrix `X`

We use `preprocess_row` for both training and validation — single source of truth,  
eliminates training/serving skew.

In [17]:
from outlier_stream import preprocess_row

train_df = pd.read_csv(TRAIN_PATH)
print(f"Training rows: {len(train_df)}")

# Build X by applying preprocess_row to every training row — same pipeline as inference
X = pd.concat(
    train_df.apply(lambda row: preprocess_row(row), axis=1).tolist(),
    ignore_index=True
)

print(f"X shape: {X.shape}")
X.head()

Training rows: 8696
X shape: (8696, 4)


,salary_processed,employment_type,has_company_logo,required_education
0,92500.0,1,1,2
1,44000.0,1,0,9
2,44000.0,1,1,9
3,44000.0,1,1,1
4,44000.0,3,1,4


## 3. Load Validation Data, Fix `t`/`f` Bug & Build `X_val`

`val.csv` stores binary columns as strings `'t'`/`'f'` instead of `1`/`0`.  
We fix this before passing rows to `preprocess_row`, which calls `int(row['has_company_logo'])` and would raise a `ValueError` otherwise.

In [18]:
val_df = pd.read_csv(VAL_PATH)
print(f"Validation rows: {len(val_df)}")

# Fix: map 't'/'f' strings to 1/0 for all binary columns
BOOL_MAP = {'t': 1, 'true': 1, '1': 1, 'f': 0, 'false': 0, '0': 0}
BINARY_COLS = ['has_company_logo', 'telecommuting', 'fraudulent']

for col in BINARY_COLS:
    val_df[col] = val_df[col].astype(str).str.lower().map(BOOL_MAP)

print("Sample after fix:")
val_df[BINARY_COLS].head(3)

Validation rows: 1877
Sample after fix:


,has_company_logo,telecommuting,fraudulent
0,1,0,0
1,1,0,0
2,1,0,0


In [19]:
# Build X_val using the same preprocess_row pipeline
X_val = pd.concat(
    val_df.apply(lambda row: preprocess_row(row), axis=1).tolist(),
    ignore_index=True
)

# Ground-truth fraud labels for evaluation
y_val = val_df['fraudulent'].astype(int)

print(f"X_val shape: {X_val.shape}")
print(f"y_val fraud rate: {y_val.mean():.2%}  ({y_val.sum()} fraudulent / {len(y_val)} total)")
X_val.head()

X_val shape: (1877, 4)
y_val fraud rate: 4.85%  (91 fraudulent / 1877 total)


,salary_processed,employment_type,has_company_logo,required_education
0,44000.0,1,1,2
1,100000.0,1,1,10
2,44000.0,1,1,9
3,44000.0,1,1,9
4,15000.0,1,0,2


## 4. Isolation Forest — Contamination Tuning

For each `contamination` value we:
1. Fit a fresh `IsolationForest` on `X` (training data)
2. Predict on `X_val`
3. Convert sklearn's output: `-1 → 1` (anomaly = fraud), `1 → 0` (normal)
4. Score against `y_val` using F1, Precision, and Recall

In [20]:
CONTAMINATION_VALUES = [0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.10]

if_results = []

for c in CONTAMINATION_VALUES:
    model_if = IsolationForest(n_estimators=100, contamination=c, random_state=42)
    model_if.fit(X)

    # sklearn outputs: -1 = anomaly, 1 = normal
    # We need:         1  = fraud,   0 = normal  — so flip the sign
    raw_pred  = model_if.predict(X_val)
    y_pred    = (raw_pred == -1).astype(int)

    if_results.append({
        'contamination': c,
        'f1':        round(f1_score(y_val, y_pred, zero_division=0), 4),
        'precision': round(precision_score(y_val, y_pred, zero_division=0), 4),
        'recall':    round(recall_score(y_val, y_pred, zero_division=0), 4),
        'flagged':   int(y_pred.sum()),
    })
    print(f"  c={c:.2f}  F1={if_results[-1]['f1']:.4f}  "
          f"P={if_results[-1]['precision']:.4f}  R={if_results[-1]['recall']:.4f}")

if_df = pd.DataFrame(if_results)
print("\nIsolation Forest Results:")
if_df

  c=0.01  F1=0.0190  P=0.0714  R=0.0110
  c=0.02  F1=0.0500  P=0.1034  R=0.0330
  c=0.03  F1=0.1119  P=0.1538  R=0.0879
  c=0.04  F1=0.1266  P=0.1493  R=0.1099
  c=0.05  F1=0.1136  P=0.1176  R=0.1099
  c=0.07  F1=0.1043  P=0.0917  R=0.1209
  c=0.10  F1=0.1778  P=0.1341  R=0.2637

Isolation Forest Results:


,contamination,f1,precision,recall,flagged
0,0.01,0.0190,0.0714,0.0110,14
1,0.02,0.0500,0.1034,0.0330,29
2,0.03,0.1119,0.1538,0.0879,52
3,0.04,0.1266,0.1493,0.1099,67
4,0.05,0.1136,0.1176,0.1099,85
5,0.07,0.1043,0.0917,0.1209,120
6,0.10,0.1778,0.1341,0.2637,179


## 5. Local Outlier Factor — Contamination Tuning

Same loop for LOF. `novelty=True` is required so that `.predict()` works on  
unseen data (`X_val`) rather than only the training set.

In [21]:
lof_results = []

for c in CONTAMINATION_VALUES:
    model_lof = LocalOutlierFactor(n_neighbors=20, contamination=c, novelty=True)
    model_lof.fit(X)

    raw_pred = model_lof.predict(X_val)
    y_pred   = (raw_pred == -1).astype(int)

    lof_results.append({
        'contamination': c,
        'f1':        round(f1_score(y_val, y_pred, zero_division=0), 4),
        'precision': round(precision_score(y_val, y_pred, zero_division=0), 4),
        'recall':    round(recall_score(y_val, y_pred, zero_division=0), 4),
        'flagged':   int(y_pred.sum()),
    })
    print(f"  c={c:.2f}  F1={lof_results[-1]['f1']:.4f}  "
          f"P={lof_results[-1]['precision']:.4f}  R={lof_results[-1]['recall']:.4f}")

lof_df = pd.DataFrame(lof_results)
print("\nLOF Results:")
lof_df

/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


  c=0.01  F1=0.0187  P=0.0625  R=0.0110
  c=0.02  F1=0.0308  P=0.0513  R=0.0220


/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


  c=0.03  F1=0.0438  P=0.0652  R=0.0330
  c=0.04  F1=0.0510  P=0.0606  R=0.0440


/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


  c=0.05  F1=0.0670  P=0.0682  R=0.0659
  c=0.07  F1=0.0639  P=0.0547  R=0.0769
  c=0.10  F1=0.0867  P=0.0622  R=0.1429

LOF Results:


/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


,contamination,f1,precision,recall,flagged
0,0.01,0.0187,0.0625,0.0110,16
1,0.02,0.0308,0.0513,0.0220,39
2,0.03,0.0438,0.0652,0.0330,46
3,0.04,0.0510,0.0606,0.0440,66
4,0.05,0.0670,0.0682,0.0659,88
5,0.07,0.0639,0.0547,0.0769,128
6,0.10,0.0867,0.0622,0.1429,209


## 5b. LOF — `n_neighbors` Sweep

LOF is sensitive to `n_neighbors`: too small and the local density estimate is noisy;  
too large and it smooths over genuine local anomalies.

We fix `contamination` at the best value found in Section 5 and sweep `n_neighbors`  
across `[10, 15, 20, 30]`. The best `(contamination, n_neighbors)` pair updates  
`best_lof` so Section 6 inherits the fully-optimised result.

In [22]:
N_NEIGHBORS_VALUES = [10, 15, 20, 30]

# Derive best contamination directly from lof_df (already in scope from Section 5)
best_lof_contamination = lof_df.loc[lof_df['f1'].idxmax(), 'contamination']
print(f"Sweeping n_neighbors with contamination fixed at {best_lof_contamination}\n")

lof_nn_results = []

for nn in N_NEIGHBORS_VALUES:
    m = LocalOutlierFactor(
        n_neighbors=nn,
        contamination=best_lof_contamination,
        novelty=True
    )
    m.fit(X)

    raw_pred = m.predict(X_val)
    y_pred   = (raw_pred == -1).astype(int)

    lof_nn_results.append({
        'n_neighbors':   nn,
        'contamination': best_lof_contamination,
        'f1':            round(f1_score(y_val, y_pred, zero_division=0), 4),
        'precision':     round(precision_score(y_val, y_pred, zero_division=0), 4),
        'recall':        round(recall_score(y_val, y_pred, zero_division=0), 4),
        'flagged':       int(y_pred.sum()),
    })
    print(f"  n_neighbors={nn:<4}  F1={lof_nn_results[-1]['f1']:.4f}  "
          f"P={lof_nn_results[-1]['precision']:.4f}  R={lof_nn_results[-1]['recall']:.4f}")

lof_nn_df = pd.DataFrame(lof_nn_results)

# Update best_lof with the optimal n_neighbors result so Section 6 uses it
best_nn_row = lof_nn_df.loc[lof_nn_df['f1'].idxmax()].to_dict()
best_lof = {
    'contamination': best_nn_row['contamination'],
    'LOF_f1':        best_nn_row['f1'],
    'LOF_precision': best_nn_row['precision'],
    'LOF_recall':    best_nn_row['recall'],
    'LOF_flagged':   best_nn_row['flagged'],
    'n_neighbors':   int(best_nn_row['n_neighbors']),
}

print(f"\nBest LOF after n_neighbors sweep → "
      f"n_neighbors={best_lof['n_neighbors']}, "
      f"contamination={best_lof['contamination']}, "
      f"F1={best_lof['LOF_f1']:.4f}")

lof_nn_df.style \
    .highlight_max(subset=['f1'], color='#c6efce') \
    .format(precision=4)

Sweeping n_neighbors with contamination fixed at 0.1



/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


  n_neighbors=10    F1=0.0993  P=0.0733  R=0.1538
  n_neighbors=15    F1=0.1018  P=0.0761  R=0.1538


/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


  n_neighbors=20    F1=0.0867  P=0.0622  R=0.1429


/Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


  n_neighbors=30    F1=0.0586  P=0.0440  R=0.0879

Best LOF after n_neighbors sweep → n_neighbors=15, contamination=0.1, F1=0.1018


,n_neighbors,contamination,f1,precision,recall,flagged
0,10,0.1000,0.0993,0.0733,0.1538,191
1,15,0.1000,0.1018,0.0761,0.1538,184
2,20,0.1000,0.0867,0.0622,0.1429,209
3,30,0.1000,0.0586,0.0440,0.0879,182


## 6. Side-by-Side Comparison

Merge both result tables and highlight the best F1 for each model.

In [23]:
comparison_df = if_df.rename(columns={
    'f1': 'IF_f1', 'precision': 'IF_precision', 'recall': 'IF_recall', 'flagged': 'IF_flagged'
}).merge(
    lof_df.rename(columns={
        'f1': 'LOF_f1', 'precision': 'LOF_precision', 'recall': 'LOF_recall', 'flagged': 'LOF_flagged'
    }),
    on='contamination'
)

best_if  = comparison_df.loc[comparison_df['IF_f1'].idxmax()]
best_lof = comparison_df.loc[comparison_df['LOF_f1'].idxmax()]

print(f"Best IsolationForest  → contamination={best_if['contamination']}  F1={best_if['IF_f1']}")
print(f"Best LOF              → contamination={best_lof['contamination']}  F1={best_lof['LOF_f1']}")
print()

comparison_df.style \
    .highlight_max(subset=['IF_f1'],  color='#c6efce') \
    .highlight_max(subset=['LOF_f1'], color='#c6efce') \
    .format(precision=4)

Best IsolationForest  → contamination=0.1  F1=0.1778
Best LOF              → contamination=0.1  F1=0.0867



,contamination,IF_f1,IF_precision,IF_recall,IF_flagged,LOF_f1,LOF_precision,LOF_recall,LOF_flagged
0,0.0100,0.0190,0.0714,0.0110,14,0.0187,0.0625,0.0110,16
1,0.0200,0.0500,0.1034,0.0330,29,0.0308,0.0513,0.0220,39
2,0.0300,0.1119,0.1538,0.0879,52,0.0438,0.0652,0.0330,46
3,0.0400,0.1266,0.1493,0.1099,67,0.0510,0.0606,0.0440,66
4,0.0500,0.1136,0.1176,0.1099,85,0.0670,0.0682,0.0659,88
5,0.0700,0.1043,0.0917,0.1209,120,0.0639,0.0547,0.0769,128
6,0.1000,0.1778,0.1341,0.2637,179,0.0867,0.0622,0.1429,209


## 7. Model Comparison & Recommendation (B-6)

We compare the two best models head-to-head across all three metrics and make a  
recommendation for which model to export as the production outlier detector.

**Guiding principle:** In fraud detection, **Recall** matters most — a missed fraud  
(false negative) is more costly than a false alarm. We use F1 as the primary ranking  
metric since it balances precision and recall, but we factor in recall when models are close.

In [24]:
# ── Head-to-head at each model's best contamination ──────────────────────────
print("=" * 60)
print("  B-6 MODEL COMPARISON: IsolationForest vs. LOF")
print("=" * 60)

metrics = ['f1', 'precision', 'recall']

print(f"\n{'Metric':<12} {'IsolationForest':>18} {'LOF':>10}  {'Winner':>10}")
print("-" * 55)
for m in metrics:
    if_val  = best_if[f'IF_{m}']
    lof_val = best_lof[f'LOF_{m}']
    winner  = 'IF' if if_val > lof_val else ('LOF' if lof_val > if_val else 'Tie')
    print(f"  {m.capitalize():<10} {if_val:>18.4f} {lof_val:>10.4f}  {winner:>10}")

print(f"\n  Best IF  contamination : {best_if['contamination']}")
print(f"  Best LOF contamination : {best_lof['contamination']}")

# ── Recommendation logic ──────────────────────────────────────────────────────
f1_delta = best_if['IF_f1'] - best_lof['LOF_f1']

if abs(f1_delta) < 0.005:
    # Models are within 0.5 F1 points — recall is the tiebreaker
    if best_if['IF_recall'] >= best_lof['LOF_recall']:
        BEST_MODEL_NAME = 'IsolationForest'
        BEST_CONTAMINATION = best_if['contamination']
        reason = "F1 scores are near-equal; IsolationForest wins on Recall."
    else:
        BEST_MODEL_NAME = 'LOF'
        BEST_CONTAMINATION = best_lof['contamination']
        reason = "F1 scores are near-equal; LOF wins on Recall."
elif f1_delta > 0:
    BEST_MODEL_NAME = 'IsolationForest'
    BEST_CONTAMINATION = best_if['contamination']
    reason = f"IsolationForest leads LOF by {f1_delta:.4f} F1 points."
else:
    BEST_MODEL_NAME = 'LOF'
    BEST_CONTAMINATION = best_lof['contamination']
    reason = f"LOF leads IsolationForest by {abs(f1_delta):.4f} F1 points."

print("\n" + "=" * 60)
print(f"  RECOMMENDATION: {BEST_MODEL_NAME}")
print(f"  Best contamination   : {BEST_CONTAMINATION}")
print(f"  Reason               : {reason}")
print("=" * 60)

  B-6 MODEL COMPARISON: IsolationForest vs. LOF

Metric          IsolationForest        LOF      Winner
-------------------------------------------------------
  F1                     0.1778     0.0867          IF
  Precision              0.1341     0.0622          IF
  Recall                 0.2637     0.1429          IF

  Best IF  contamination : 0.1
  Best LOF contamination : 0.1

  RECOMMENDATION: IsolationForest
  Best contamination   : 0.1
  Reason               : IsolationForest leads LOF by 0.0911 F1 points.


## 8. Export Best Model (B-7)

Re-fit the winning model at its optimal contamination value on the full training set  
and serialize it to Google Drive `/models/outlier_forest.pkl` using `joblib`.

In [25]:
import joblib

MODEL_EXPORT_PATH = os.path.join(
    DRIVE_ROOT, "models", "outlier_forest.pkl"
)

# Re-fit the winning model on the full training set at optimal contamination
if BEST_MODEL_NAME == 'IsolationForest':
    best_model = IsolationForest(
        n_estimators=100,
        contamination=BEST_CONTAMINATION,
        random_state=42
    )
else:
    best_model = LocalOutlierFactor(
        n_neighbors=20,
        contamination=BEST_CONTAMINATION,
        novelty=True
    )

best_model.fit(X)
print(f"Fitted {BEST_MODEL_NAME} (contamination={BEST_CONTAMINATION}) on {X.shape[0]} training rows.")

# Serialize to Drive
joblib.dump(best_model, MODEL_EXPORT_PATH)

# Verify
assert os.path.exists(MODEL_EXPORT_PATH), "Export failed — file not found!"
size_kb = os.path.getsize(MODEL_EXPORT_PATH) / 1024
print(f"\nExported → {MODEL_EXPORT_PATH}")
print(f"File size: {size_kb:.1f} KB")

Fitted IsolationForest (contamination=0.1) on 8696 training rows.

Exported → /Users/isaganijulian/Documents/GitHub/Guardian-Recruit-Fraud-Detection/models/outlier_forest.pkl
File size: 780.2 KB
